# Customer Churn Prediction & Analytics
**Software Intelligence Lab Project | Google Colab + Streamlit**

This notebook uses the provided `Telco-Customer-Churn.csv` dataset. It brings the concepts from the uploaded SI practicals into one customer-churn project: web scraping basics, data distributions, sampling and point estimation, confidence intervals, outlier detection with IQR, hypothesis tests, correlation/covariance, encoding and scaling, regression model comparison for a numeric business variable, and classification models for churn prediction.

**Target:** `Churn` (`Yes` / `No`). The Streamlit app is a separate interface in `app.py`; this notebook performs the full analysis and exports a trained model bundle.

In [ ]:
# 1. Install/import libraries (Colab)
!pip -q install beautifulsoup4 requests scikit-learn scipy seaborn joblib

import os, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.inspection import permutation_importance
import joblib
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## 2. Load the customer churn data
In Colab, upload `Telco-Customer-Churn.csv` when prompted. The CSV is also included in this project ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select Telco-Customer-Churn.csv
DATA_FILE = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if DATA_FILE is None:
    raise ValueError("Please upload the Telco-Customer-Churn.csv file.")
df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
display(df.head())
print(df.info())

## 3. Practical: web scraping fundamentals
The original practical used `requests` and BeautifulSoup to parse a public HTML page. This section demonstrates the same technique on a small public demo page; the churn analysis itself uses the provided CSV, not scraped customer data. If the website is unavailable, skip this cell and continue.

In [ ]:
try:
    url = "https://www.scrapethissite.com/pages/simple/"
    response = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    countries = soup.select(".country")
    scraped_demo = pd.DataFrame({
        "Country": [x.select_one(".country-name").get_text(strip=True) for x in countries],
        "Capital": [x.select_one(".country-capital").get_text(strip=True) for x in countries],
        "Population": [x.select_one(".country-population").get_text(strip=True) for x in countries]
    })
    display(scraped_demo.head())
except Exception as e:
    print("Demo website could not be reached; scraping concept shown in code. Continue with the supplied CSV.")
    print(type(e).__name__, str(e)[:180])

## 4. Data understanding and cleaning
This applies the dataset inspection, `head`, `shape`, `info`, `describe`, missing-value checks and cleaning ideas from the practicals.

In [ ]:
print("Rows, columns:", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing"))
print("\nSummary of numeric columns:")
display(df.describe().T)
print("\nChurn counts:")
display(df["Churn"].value_counts(dropna=False).to_frame("Customers"))

In [ ]:
# TotalCharges is stored as text in the source CSV; convert it to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["SeniorCitizen"] = df["SeniorCitizen"].astype(str)
df["Churn"] = df["Churn"].astype(str).str.strip()
df = df.dropna(subset=["Churn"]).copy()
print("Missing values after TotalCharges conversion:")
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("Missing"))

## 5. Practical: distributions and exploratory visualisation
Use histograms and count plots to understand the shape of numeric variables and the distribution of churn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["tenure"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Tenure")
sns.histplot(df["MonthlyCharges"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribution of Monthly Charges")
sns.countplot(data=df, x="Churn", ax=axes[2])
axes[2].set_title("Churn Class Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.histplot(data=df, x="MonthlyCharges", hue="Churn", bins=30, element="step", stat="density", common_norm=False)
plt.title("Monthly Charges by Churn Status")
plt.show()

## 6. Practical: sampling distribution, point estimate and confidence interval
Repeatedly sample customer monthly charges and compare the sample mean with the population mean. Then calculate a 95% confidence interval for the mean monthly charge.

In [ ]:
population = df["MonthlyCharges"].dropna().to_numpy()
population_mean = population.mean()
population_std = population.std(ddof=1)
sample_size = 30
number_of_samples = 1000
sample_means = []
rng = np.random.default_rng(RANDOM_STATE)
for _ in range(number_of_samples):
    sample = rng.choice(population, size=sample_size, replace=True)
    sample_means.append(sample.mean())
sample_means = np.array(sample_means)
standard_error = population_std / np.sqrt(sample_size)
sample_mean = sample_means.mean()
ci_low, ci_high = stats.t.interval(0.95, df=len(population)-1, loc=population_mean,
                                   scale=stats.sem(population))
print(f"Population mean monthly charge: ${population_mean:.2f}")
print(f"Average of repeated sample means: ${sample_mean:.2f}")
print(f"Standard error for n={sample_size}: ${standard_error:.2f}")
print(f"95% confidence interval for mean monthly charge: (${ci_low:.2f}, ${ci_high:.2f})")
plt.figure(figsize=(8, 4))
plt.hist(sample_means, bins=30)
plt.axvline(population_mean, linestyle="--", label="Population mean")
plt.title("Sampling Distribution of Mean Monthly Charges")
plt.xlabel("Sample mean ($)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## 7. Practical: outlier detection using IQR
Use the IQR rule on `MonthlyCharges` and `TotalCharges`. A statistical outlier is not automatically a bad customer record, so we flag and inspect rather than deleting rows blindly.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["MonthlyCharges", "TotalCharges"]:
    values = df[col].dropna()
    lower, upper = iqr_bounds(values)
    count = ((values < lower) | (values > upper)).sum()
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}, flagged rows={count}")
    plt.figure(figsize=(7, 2.5))
    sns.boxplot(x=values)
    plt.title(f"IQR Boxplot: {col}")
    plt.show()

## 8. Practical: hypothesis testing
Question: Is average monthly charge different for customers who churn and customers who stay?  
**H0:** The two groups have equal means. **H1:** The means differ. Use Welch's independent t-test and significance level 0.05. This tests association in this dataset; it does not prove causation.

In [ ]:
churn_yes = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].dropna()
churn_no = df.loc[df["Churn"] == "No", "MonthlyCharges"].dropna()
t_stat, p_value = stats.ttest_ind(churn_yes, churn_no, equal_var=False)
print("Mean monthly charge (churn Yes):", round(churn_yes.mean(), 2))
print("Mean monthly charge (churn No):", round(churn_no.mean(), 2))
print("t-statistic:", round(t_stat, 4), "p-value:", p_value)
if p_value < 0.05:
    print("Decision: Reject H0. The group means differ significantly at alpha=0.05.")
else:
    print("Decision: Fail to reject H0. Evidence of a mean difference is insufficient.")

## 8B. Practical: Z-test and one-way ANOVA
The uploaded statistics practical includes Z-test and ANOVA concepts. Here, a one-sample Z-test compares mean monthly charges against an illustrative reference value of $70, using the sample standard deviation as an approximation; ANOVA compares mean monthly charges across contract types. The $70 reference is a demonstration benchmark, not an official industry standard.


In [ ]:
from scipy import stats
charges = df['MonthlyCharges'].dropna().to_numpy()
reference_mean = 70.0
z_stat = (charges.mean() - reference_mean) / (charges.std(ddof=1) / np.sqrt(len(charges)))
z_p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print('One-sample Z-test: H0 mean monthly charge = $70')
print('Sample mean:', round(charges.mean(), 2))
print('Z statistic:', round(z_stat, 4), 'p-value:', z_p_value)
print('Decision:', 'Reject H0' if z_p_value < 0.05 else 'Fail to reject H0', '(alpha = 0.05)')

if 'Contract' in df.columns:
    groups = [g['MonthlyCharges'].dropna().to_numpy() for _, g in df.groupby('Contract')]
    groups = [g for g in groups if len(g) > 0]
    f_stat, anova_p = stats.f_oneway(*groups)
    print('\nOne-way ANOVA: mean MonthlyCharges across Contract groups')
    print('F statistic:', round(f_stat, 4), 'p-value:', anova_p)
    print('Decision:', 'Reject H0: at least one group mean differs' if anova_p < 0.05 else 'Fail to reject H0')


## 9. Practical: covariance, Pearson and Spearman correlation
Explore relationships among numeric customer variables. Correlation is not causation, and a correlation coefficient alone is not a churn prediction.

In [ ]:
numeric = df[["tenure", "MonthlyCharges", "TotalCharges"]].copy()
print("Covariance matrix:")
display(numeric.cov())
print("Pearson correlation matrix:")
display(numeric.corr(method="pearson"))
print("Spearman correlation matrix:")
display(numeric.corr(method="spearman"))
plt.figure(figsize=(7, 5))
sns.heatmap(numeric.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.show()
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df.sample(min(1500, len(df)), random_state=RANDOM_STATE),
                x="tenure", y="TotalCharges", hue="Churn", alpha=0.65)
plt.title("Tenure vs Total Charges")
plt.show()

## 10. Practical: encoding, scaling and train/test split
Categorical variables are one-hot encoded; numeric variables are imputed and standardized. The preprocessing is fitted only on the training data through a Pipeline, preventing test-data leakage. `customerID` is an identifier and is excluded from model features.

In [ ]:
TARGET = "Churn"
DROP_COLS = ["customerID", TARGET]
X = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()
y = (df[TARGET].str.lower() == "yes").astype(int)
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 11. Churn classification: Logistic Regression, Decision Tree and Random Forest
Churn is a classification task, so classification algorithms and metrics are used here. The regression models in the next section are a separate extension to apply the regression practical to a numeric business variable.

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=12, min_samples_leaf=2,
                                            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
}
classification_results = []
fitted_pipelines = {}
for name, estimator in classifiers.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    }
    classification_results.append(row)
    fitted_pipelines[name] = pipe
results_df = pd.DataFrame(classification_results).sort_values("F1", ascending=False)
display(results_df.round(4))

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_pipelines[best_name]
best_pred = best_model.predict(X_test)
print("Best model selected by F1:", best_name)
print(classification_report(y_test, best_pred, target_names=["Stayed", "Churned"], zero_division=0))
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted stayed", "Predicted churned"],
            yticklabels=["Actual stayed", "Actual churned"])
plt.title(f"Confusion Matrix — {best_name}")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()
results_plot = results_df.set_index("Model")[["Precision", "Recall", "F1"]]
results_plot.plot(kind="bar", figsize=(9, 4), ylim=(0, 1), title="Churn Model Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 12. Practical: regression model comparison (secondary analysis)
The churn target is categorical, so regression is not appropriate for predicting churn. To apply the regression algorithms from the practical meaningfully, this section predicts numeric `TotalCharges` using `tenure` and `MonthlyCharges`. It is a secondary business analysis, not the churn classifier.

In [ ]:
reg_df = df[["tenure", "MonthlyCharges", "TotalCharges"]].dropna()
Xr = reg_df[["tenure", "MonthlyCharges"]]
yr = reg_df["TotalCharges"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.20, random_state=RANDOM_STATE
)
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0, max_iter=5000),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
}
reg_rows = []
for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    reg_rows.append({"Model": name,
                     "MAE": mean_absolute_error(yr_test, pred),
                     "MSE": mean_squared_error(yr_test, pred),
                     "RMSE": np.sqrt(mean_squared_error(yr_test, pred)),
                     "R2 Score": r2_score(yr_test, pred)})
reg_results = pd.DataFrame(reg_rows).sort_values("RMSE")
display(reg_results.round(3))
reg_results.plot(x="Model", y="R2 Score", kind="bar", legend=False, figsize=(9, 4), title="Regression Model R² Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 13. Model interpretation and export for Streamlit
The app needs the entire preprocessing + model pipeline, so we export the selected pipeline as one file. The app also includes a model-based churn probability and a simple low/medium/high risk band; the band is a user-facing grouping, not a calibrated business policy.

In [ ]:
model_bundle = {
    "model": best_model,
    "model_name": best_name,
    "feature_columns": X.columns.tolist(),
    "target_name": TARGET
}
joblib.dump(model_bundle, "churn_model_bundle.pkl")
df.to_csv("cleaned_churn_data.csv", index=False)
print("Saved: churn_model_bundle.pkl and cleaned_churn_data.csv")
from google.colab import files
files.download("churn_model_bundle.pkl")
files.download("cleaned_churn_data.csv")

## 14. Optional: one-customer prediction example
This uses a real row from the held-out feature table as an example. For a new customer, provide the same feature columns through the Streamlit form.

In [ ]:
example_customer = X_test.iloc[[0]].copy()
example_probability = best_model.predict_proba(example_customer)[0, 1]
print("Selected model:", best_name)
print("Predicted churn probability:", f"{example_probability:.1%}")
print("Predicted class:", "Churn" if example_probability >= 0.5 else "No churn")
display(example_customer)

## Conclusion
This project applies the SI practical concepts to telecom churn data. Statistical tests and correlations are exploratory; the classification model provides a probability estimate, not certainty. Model performance should be judged using recall, precision, F1 and ROC-AUC together, because missing a customer who may churn and contacting a customer who will stay have different business costs.